In [ ]:
#Importing libraries
import itertools
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
import scipy as sp 
import sklearn
import scipy.stats as st
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score
from sklearn.metrics import confusion_matrix
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import RFE
from sklearn.svm import SVC
from sklearn.model_selection import train_test_split
from sklearn.model_selection import GridSearchCV
from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.model_selection import KFold
from sklearn.model_selection import StratifiedKFold
from sklearn.model_selection import cross_val_score
from numpy import mean
from numpy import std
from matplotlib import pyplot

#SVC met BA threshold for all sources so below are the RFE results for them

### AFFF-GW

In [ ]:
data_rf = pd.read_csv(r'240905-NTA-Paper2-AnnotatedFeatures-ONLY-LogT-Input.csv', header=0) #Targets: 92 samples X 581 features
#del data_rf[data_rf.columns[0]] #Dropping sample information
#print(data_rf)

In [ ]:
#Prompt user for source type of interest (AFFF-GW, LF, BSL, WWTP, PP or PG)
preferred_type = input("Enter the source type of interest: ")

In [ ]:
#Manipulating data frame based on user input to make "Type" column read 1 for all samples of source of interest and 0 for all other samples
#Set up for binary classification (one-vs-all format)

# Define a function to apply to each row
def set_type(row):
    if row['Type'] == preferred_type:
        return 1
    else:
        return 0

# Create a new column "Type 2" with the updated values
data_rf['Type_2'] = data_rf.apply(set_type, axis=1)
del data_rf[data_rf.columns[0]] #Dropping original type column
#Reordering columns with Type_2 as first column
cols = list(data_rf.columns)
cols = [cols[-1]] + cols[:-1]
data_rf = data_rf[cols]

# Save the updated DataFrame to a new CSV file (if needed)
data_rf.to_csv('sample_data_with_labels_NEW10.csv', index=False)

In [ ]:
#Changing pandas data frame to numpy for use in ML
data_rf_np = data_rf.to_numpy()
target_1 = data_rf_np[:,0].reshape(-1,1) #Convert target variables to 2D-array for sci-kit learn
data_1 = data_rf_np[:,1:]

#class_names=np.array([0.0,1.0])
#print(data_1.shape)
print(data_1)
#data_1 = pd.DataFrame(data_1)
#data_1.to_csv('log10_dat.csv', index=False)

In [ ]:
#GW with SVC
from sklearn.svm import SVC
# get a list of models to evaluate
def get_models():
    models = dict()
    for i in range(1, 15):
        rfe = RFE(estimator=SVC(kernel='linear',
                    C=92.13002, 
                    
                    tol=0.00001, 
                    shrinking=True, 
                    cache_size=200, 
                    verbose=False, 
                    max_iter=-1, 
                    probability=True), n_features_to_select=i)
        model = SVC(kernel='linear',
                    C=92.13002, 
                    
                    tol=0.00001, 
                    shrinking=True, 
                    cache_size=200, 
                    verbose=False, 
                    max_iter=-1, 
                    probability=True)
        models[str(i)] = Pipeline(steps=[('s',rfe),('m',model)])
    return models

#Evaluate model
def evaluate_model(model, X, y):
    cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
    scores = cross_val_score(model, data_1, target_1, scoring='balanced_accuracy', cv=cv, n_jobs=-1, error_score='raise')
    return scores

In [ ]:
# Get the models to evaluate
models = get_models()

# evaluate the models and store results
results, names = list(), list()
for name, model in models.items():
    scores = evaluate_model(model, data_1, target_1)
    results.append(scores)
    names.append(name)
    print('>%s Features: %d, Balanced Accuracy: %.3f (%.3f)' % (name, int(name), np.mean(scores), np.std(scores)))

In [ ]:
rfe = RFE(estimator=SVC(kernel='linear',
                    C=92.13002, 
                    
                    tol=0.00001, 
                    shrinking=True, 
                    cache_size=200, 
                    verbose=False, 
                    max_iter=-1, 
                    probability=True), n_features_to_select=14)
rfe.fit(data_1,target_1.ravel())
for i in range(data_1.shape[1]):
 print('Column: %d, Selected %s, Rank: %.3f' % (i, rfe.support_[i], rfe.ranking_[i]))

In [ ]:
# Load feature names from the second dataset
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

# Print selected features with their names
selected_feature_indices = [i for i in range(len(rfe.support_)) if rfe.support_[i]]

for index in selected_feature_indices:
    print('Feature Name: %s, Index: %d, Rank: %.3f' % (feature_names[index], index, rfe.ranking_[index]))

In [ ]:
selected_features_df = pd.DataFrame({
    'Feature Name': [feature_names[index] for index in selected_feature_indices]
})

# Export the DataFrame to an Excel file
selected_features_df.to_excel("RFE_selected_features_norm_GW_SVC.xlsx", index=False)

### LL

In [ ]:
data_rf = pd.read_csv(r'240905-NTA-Paper2-AnnotatedFeatures-ONLY-LogT-Input.csv', header=0) #Targets: 92 samples X 581 features
#del data_rf[data_rf.columns[0]] #Dropping sample information
#print(data_rf)

In [ ]:
#Prompt user for source type of interest (AFFF-GW, LF, BSL, WWTP, PP or PG)
preferred_type = input("Enter the source type of interest: ")

In [ ]:
#Manipulating data frame based on user input to make "Type" column read 1 for all samples of source of interest and 0 for all other samples
#Set up for binary classification (one-vs-all format)

# Define a function to apply to each row
def set_type(row):
    if row['Type'] == preferred_type:
        return 1
    else:
        return 0

# Create a new column "Type 2" with the updated values
data_rf['Type_2'] = data_rf.apply(set_type, axis=1)
del data_rf[data_rf.columns[0]] #Dropping original type column
#Reordering columns with Type_2 as first column
cols = list(data_rf.columns)
cols = [cols[-1]] + cols[:-1]
data_rf = data_rf[cols]

# Save the updated DataFrame to a new CSV file (if needed)
data_rf.to_csv('sample_data_with_labels_NEW10.csv', index=False)

In [ ]:
#Changing pandas data frame to numpy for use in ML
data_rf_np = data_rf.to_numpy()
target_1 = data_rf_np[:,0].reshape(-1,1) #Convert target variables to 2D-array for sci-kit learn
data_1 = data_rf_np[:,1:]

#class_names=np.array([0.0,1.0])
#print(data_1.shape)
print(data_1)
#data_1 = pd.DataFrame(data_1)
#data_1.to_csv('log10_dat.csv', index=False)

In [ ]:
#ll with SVC
from sklearn.svm import SVC
# get a list of models to evaluate
def get_models():
    models = dict()
    for i in range(1, 31):
        rfe = RFE(estimator=SVC(kernel='linear',
                    C=7.05191, 
                    
                    tol=0.00001, 
                    shrinking=True, 
                    cache_size=200, 
                    verbose=False, 
                    max_iter=-1, 
                    probability=True), n_features_to_select=i)
        model = SVC(kernel='linear',
                    C=7.05191, 
                    
                    tol=0.00001, 
                    shrinking=True, 
                    cache_size=200, 
                    verbose=False, 
                    max_iter=-1, 
                    probability=True)
        models[str(i)] = Pipeline(steps=[('s',rfe),('m',model)])
    return models

#Evaluate model
def evaluate_model(model, X, y):
    cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
    scores = cross_val_score(model, data_1, target_1, scoring='balanced_accuracy', cv=cv, n_jobs=-1, error_score='raise')
    return scores

In [ ]:
# Get the models to evaluate
models = get_models()

# evaluate the models and store results
results, names = list(), list()
for name, model in models.items():
    scores = evaluate_model(model, data_1, target_1)
    results.append(scores)
    names.append(name)
    print('>%s Features: %d, Balanced Accuracy: %.3f (%.3f)' % (name, int(name), np.mean(scores), np.std(scores)))

In [ ]:
rfe = RFE(estimator=SVC(kernel='linear',
                    C=7.05191, 
                    
                    tol=0.00001, 
                    shrinking=True, 
                    cache_size=200, 
                    verbose=False, 
                    max_iter=-1, 
                    probability=True), n_features_to_select=10)
rfe.fit(data_1,target_1.ravel())
for i in range(data_1.shape[1]):
 print('Column: %d, Selected %s, Rank: %.3f' % (i, rfe.support_[i], rfe.ranking_[i]))

In [ ]:
# Load feature names from the second dataset
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

# Print selected features with their names
selected_feature_indices = [i for i in range(len(rfe.support_)) if rfe.support_[i]]

for index in selected_feature_indices:
    print('Feature Name: %s, Index: %d, Rank: %.3f' % (feature_names[index], index, rfe.ranking_[index]))

In [ ]:
selected_features_df = pd.DataFrame({
    'Feature Name': [feature_names[index] for index in selected_feature_indices]
})

# Export the DataFrame to an Excel file
selected_features_df.to_excel("RFE_selected_features_norm_LL_SVC.xlsx", index=False)

### PP

In [ ]:
data_rf = pd.read_csv(r'240905-NTA-Paper2-AnnotatedFeatures-ONLY-LogT-Input.csv', header=0) #Targets: 92 samples X 581 features
#del data_rf[data_rf.columns[0]] #Dropping sample information
#print(data_rf)

In [ ]:
#Prompt user for source type of interest (AFFF-GW, LF, BSL, WWTP, PP or PG)
preferred_type = input("Enter the source type of interest: ")

In [ ]:
#Manipulating data frame based on user input to make "Type" column read 1 for all samples of source of interest and 0 for all other samples
#Set up for binary classification (one-vs-all format)

# Define a function to apply to each row
def set_type(row):
    if row['Type'] == preferred_type:
        return 1
    else:
        return 0

# Create a new column "Type 2" with the updated values
data_rf['Type_2'] = data_rf.apply(set_type, axis=1)
del data_rf[data_rf.columns[0]] #Dropping original type column
#Reordering columns with Type_2 as first column
cols = list(data_rf.columns)
cols = [cols[-1]] + cols[:-1]
data_rf = data_rf[cols]

# Save the updated DataFrame to a new CSV file (if needed)
data_rf.to_csv('sample_data_with_labels_NEW10.csv', index=False)

In [ ]:
#Changing pandas data frame to numpy for use in ML
data_rf_np = data_rf.to_numpy()
target_1 = data_rf_np[:,0].reshape(-1,1) #Convert target variables to 2D-array for sci-kit learn
data_1 = data_rf_np[:,1:]

#class_names=np.array([0.0,1.0])
#print(data_1.shape)
print(data_1)
#data_1 = pd.DataFrame(data_1)
#data_1.to_csv('log10_dat.csv', index=False)

In [ ]:
#ll with SVC
from sklearn.svm import SVC
# get a list of models to evaluate
def get_models():
    models = dict()
    for i in range(1, 31):
        rfe = RFE(estimator=SVC(kernel='linear',
                    C=32.23429, 
                    
                    tol=0.00001, 
                    shrinking=True, 
                    cache_size=200, 
                    verbose=False, 
                    max_iter=-1, 
                    probability=True), n_features_to_select=i)
        model = SVC(kernel='linear',
                    C=32.23429, 
                    
                    tol=0.00001, 
                    shrinking=True, 
                    cache_size=200, 
                    verbose=False, 
                    max_iter=-1, 
                    probability=True)
        models[str(i)] = Pipeline(steps=[('s',rfe),('m',model)])
    return models

#Evaluate model
def evaluate_model(model, X, y):
    cv = RepeatedStratifiedKFold(n_splits=10, n_repeats=3, random_state=1)
    scores = cross_val_score(model, data_1, target_1, scoring='balanced_accuracy', cv=cv, n_jobs=-1, error_score='raise')
    return scores

In [ ]:
# Get the models to evaluate
models = get_models()

# evaluate the models and store results
results, names = list(), list()
for name, model in models.items():
    scores = evaluate_model(model, data_1, target_1)
    results.append(scores)
    names.append(name)
    print('>%s Features: %d, Balanced Accuracy: %.3f (%.3f)' % (name, int(name), np.mean(scores), np.std(scores)))

In [ ]:
rfe = RFE(estimator=SVC(kernel='linear',
                    C=32.23429, 
                    
                    tol=0.00001, 
                    shrinking=True, 
                    cache_size=200, 
                    verbose=False, 
                    max_iter=-1, 
                    probability=True), n_features_to_select=16)
rfe.fit(data_1,target_1.ravel())
for i in range(data_1.shape[1]):
 print('Column: %d, Selected %s, Rank: %.3f' % (i, rfe.support_[i], rfe.ranking_[i]))

In [ ]:
# Load feature names from the second dataset
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

# Print selected features with their names
selected_feature_indices = [i for i in range(len(rfe.support_)) if rfe.support_[i]]

for index in selected_feature_indices:
    print('Feature Name: %s, Index: %d, Rank: %.3f' % (feature_names[index], index, rfe.ranking_[index]))

In [ ]:
selected_features_df = pd.DataFrame({
    'Feature Name': [feature_names[index] for index in selected_feature_indices]
})

# Export the DataFrame to an Excel file
selected_features_df.to_excel("RFE_selected_features_norm_PP_SVC.xlsx", index=False)

### PG

In [ ]:
data_rf = pd.read_csv(r'240905-NTA-Paper2-AnnotatedFeatures-ONLY-LogT-Input.csv', header=0) #Targets: 92 samples X 581 features
#del data_rf[data_rf.columns[0]] #Dropping sample information
#print(data_rf)

In [ ]:
#Prompt user for source type of interest (AFFF-GW, LF, BSL, WWTP, PP or PG)
preferred_type = input("Enter the source type of interest: ")

In [ ]:
#Manipulating data frame based on user input to make "Type" column read 1 for all samples of source of interest and 0 for all other samples
#Set up for binary classification (one-vs-all format)

# Define a function to apply to each row
def set_type(row):
    if row['Type'] == preferred_type:
        return 1
    else:
        return 0

# Create a new column "Type 2" with the updated values
data_rf['Type_2'] = data_rf.apply(set_type, axis=1)
del data_rf[data_rf.columns[0]] #Dropping original type column
#Reordering columns with Type_2 as first column
cols = list(data_rf.columns)
cols = [cols[-1]] + cols[:-1]
data_rf = data_rf[cols]

# Save the updated DataFrame to a new CSV file (if needed)
data_rf.to_csv('sample_data_with_labels_NEW10.csv', index=False)

In [ ]:
#Changing pandas data frame to numpy for use in ML
data_rf_np = data_rf.to_numpy()
target_1 = data_rf_np[:,0].reshape(-1,1) #Convert target variables to 2D-array for sci-kit learn
data_1 = data_rf_np[:,1:]

#class_names=np.array([0.0,1.0])
#print(data_1.shape)
print(data_1)
#data_1 = pd.DataFrame(data_1)
#data_1.to_csv('log10_dat.csv', index=False)

In [ ]:
#ll with SVC
from sklearn.svm import SVC
# get a list of models to evaluate
def get_models():
    models = dict()
    for i in range(1, 31):
        rfe = RFE(estimator=SVC(kernel='linear',
                    C=88.02632, 
                    
                    tol=0.00001, 
                    shrinking=True, 
                    cache_size=200, 
                    verbose=False, 
                    max_iter=-1, 
                    probability=True), n_features_to_select=i)
        model = SVC(kernel='linear',
                    C=88.02632, 
                    
                    tol=0.00001, 
                    shrinking=True, 
                    cache_size=200, 
                    verbose=False, 
                    max_iter=-1, 
                    probability=True)
        models[str(i)] = Pipeline(steps=[('s',rfe),('m',model)])
    return models

#Evaluate model
def evaluate_model(model, X, y):
    cv = RepeatedStratifiedKFold(n_splits=4, n_repeats=3, random_state=1)
    scores = cross_val_score(model, data_1, target_1, scoring='balanced_accuracy', cv=cv, n_jobs=-1, error_score='raise')
    return scores

In [ ]:
# Get the models to evaluate
models = get_models()

# evaluate the models and store results
results, names = list(), list()
for name, model in models.items():
    scores = evaluate_model(model, data_1, target_1)
    results.append(scores)
    names.append(name)
    print('>%s Features: %d, Balanced Accuracy: %.3f (%.3f)' % (name, int(name), np.mean(scores), np.std(scores)))

In [ ]:
rfe = RFE(estimator=SVC(kernel='linear',
                    C=88.02632, 
                    
                    tol=0.00001, 
                    shrinking=True, 
                    cache_size=200, 
                    verbose=False, 
                    max_iter=-1, 
                    probability=True), n_features_to_select=5)
rfe.fit(data_1,target_1.ravel())
for i in range(data_1.shape[1]):
 print('Column: %d, Selected %s, Rank: %.3f' % (i, rfe.support_[i], rfe.ranking_[i]))

In [ ]:
# Load feature names from the second dataset
labels_dat = pd.read_csv("240905-NTA-Paper2-AnnotatedFeatures-ONLY-Labels.csv")
feature_names = labels_dat.columns

# Print selected features with their names
selected_feature_indices = [i for i in range(len(rfe.support_)) if rfe.support_[i]]

for index in selected_feature_indices:
    print('Feature Name: %s, Index: %d, Rank: %.3f' % (feature_names[index], index, rfe.ranking_[index]))

In [ ]:
selected_features_df = pd.DataFrame({
    'Feature Name': [feature_names[index] for index in selected_feature_indices]
})

# Export the DataFrame to an Excel file
selected_features_df.to_excel("RFE_selected_features_norm_PG_SVC.xlsx", index=False)